# Module 5 • Neural Networks for Natural Language Processing

# Lesson 28 • LSTM and GRU Networks for Long-Range Dependencies

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 150–190 minutes

---

## Scope

This lesson develops gated recurrent neural networks for sequence modeling.
It explains why basic RNNs struggle with long-range dependencies, derives
LSTM and GRU gates, implements individual cells with NumPy, compares parameter
counts, handles padding and packed sequences, trains executable PyTorch
classifiers, evaluates errors, and discusses Arabic and multilingual use.

All experiments run locally on CPU and require no external downloads.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain why vanilla RNNs struggle with long-range dependencies;
- describe the LSTM cell state and hidden state;
- explain the forget, input, candidate, and output gates;
- implement one LSTM step with NumPy;
- explain the GRU reset and update gates;
- implement one GRU step with NumPy;
- compare LSTM, GRU, and vanilla RNN parameter counts;
- track tensor shapes in gated sequence models;
- handle padding and sequence lengths;
- use packed sequences in PyTorch;
- build LSTM and GRU text classifiers;
- compare unidirectional and bidirectional models;
- apply dropout, gradient clipping, and early stopping;
- inspect gate behavior and hidden-state representations;
- evaluate class performance and errors;
- discuss Arabic tokenization and long-range dependencies.

## Table of Contents

1. From Vanilla RNNs to Gated RNNs
2. Long-Range Dependency Problem
3. LSTM Overview
4. Forget Gate
5. Input Gate and Candidate Memory
6. Cell-State Update
7. Output Gate and Hidden State
8. NumPy LSTM Cell
9. LSTM Shape Reasoning
10. LSTM Parameter Count
11. GRU Overview
12. Reset Gate
13. Update Gate
14. Candidate Hidden State
15. NumPy GRU Cell
16. GRU Shape Reasoning
17. GRU Parameter Count
18. LSTM Versus GRU
19. Bidirectional Gated RNNs
20. Multi-Layer Gated RNNs
21. Padding, Lengths, and Masks
22. Dataset
23. Train, Validation, and Test Splits
24. Vocabulary and Encoding
25. DataLoaders and Collation
26. LSTM Text Classifier
27. GRU Text Classifier
28. Training Utilities
29. Training the LSTM
30. Training the GRU
31. Comparing Models
32. Evaluation and Confusion Matrices
33. Error Analysis
34. Bidirectional Model
35. Representation Inspection
36. Gate Interpretation
37. Gradient Clipping and Stability
38. Dropout and Regularization
39. Computational Cost
40. Common Failure Modes
41. Arabic and Multilingual Considerations
42. Reproducibility and Reporting
43. Knowledge Check
44. Exercises
45. Summary and Next Lesson

# 1. From Vanilla RNNs to Gated RNNs

A vanilla RNN repeatedly updates a hidden state:

\[
h_t = tanh(x_tW_x + h_{t-1}W_h + b)
\]

The same recurrent transformation is applied at every time step. This
parameter sharing is efficient, but repeated nonlinear transformations can
weaken long-distance gradient flow.

In [ ]:
import copy
import math
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)
from torch.utils.data import DataLoader, Dataset

model_families = pd.DataFrame(
    [
        ("Vanilla RNN", "single hidden update", "lowest"),
        ("LSTM", "cell state + four transformations", "highest"),
        ("GRU", "hidden state + three transformations", "medium"),
    ],
    columns=["Model", "Memory mechanism", "Relative parameter cost"],
)

model_families

Gated recurrent networks learn when to retain, overwrite, expose, or reset
information.

# 2. Long-Range Dependency Problem

Consider:

```text
The report that the researchers submitted after several revisions was accepted.
```

To interpret `was`, the model may need information introduced many tokens
earlier.

In [ ]:
dependency_examples = pd.DataFrame(
    [
        (
            "Subject–verb agreement",
            "The reports ... were accepted",
        ),
        (
            "Negation",
            "The service was not satisfactory",
        ),
        (
            "Coreference",
            "Sara called Mona because she needed help",
        ),
        (
            "Discourse state",
            "Earlier evidence changes a later decision",
        ),
    ],
    columns=["Dependency", "Example"],
)

dependency_examples

Gating does not guarantee perfect long-range reasoning, but it creates
shorter and more controllable gradient paths than a basic RNN.

# 3. LSTM Overview

An LSTM maintains:

- hidden state \(h_t\);
- cell state \(c_t\).

Core equations:

\[
f_t = \sigma(x_tW_{xf} + h_{t-1}W_{hf} + b_f)
\]

\[
i_t = \sigma(x_tW_{xi} + h_{t-1}W_{hi} + b_i)
\]

\[
\tilde{c}_t = tanh(x_tW_{xc} + h_{t-1}W_{hc} + b_c)
\]

\[
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
\]

\[
o_t = \sigma(x_tW_{xo} + h_{t-1}W_{ho} + b_o)
\]

\[
h_t = o_t \odot tanh(c_t)
\]

The cell state provides an additive memory path. Gates control information
flow element by element.

# 4. Forget Gate

The forget gate determines how much previous memory to retain.

Values near:

- 1 preserve memory;
- 0 erase memory.

In [ ]:
def sigmoid(values: np.ndarray) -> np.ndarray:
    clipped = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-clipped))


gate_logits = np.array([-5.0, -1.0, 0.0, 1.0, 5.0])
forget_values = sigmoid(gate_logits)

pd.DataFrame(
    {
        "logit": gate_logits,
        "forget_gate": forget_values,
    }
)

Forget-gate bias is sometimes initialized positively so memory is initially
retained more easily.

# 5. Input Gate and Candidate Memory

The input gate controls how much new candidate memory enters the cell.

The candidate memory proposes content; the input gate scales that proposal.

In [ ]:
candidate_memory = np.array(
    [-0.9, -0.2, 0.4, 0.8]
)

input_gate = np.array(
    [0.1, 0.7, 0.5, 0.9]
)

written_memory = (
    input_gate
    * candidate_memory
)

pd.DataFrame(
    {
        "candidate": candidate_memory,
        "input_gate": input_gate,
        "written_memory": written_memory,
    }
)

# 6. Cell-State Update

The updated memory combines retained old information and gated new
information.

In [ ]:
previous_cell = np.array(
    [0.8, -0.5, 0.2, 0.9]
)

forget_gate = np.array(
    [0.9, 0.2, 0.8, 0.4]
)

new_cell = (
    forget_gate
    * previous_cell
    + written_memory
)

pd.DataFrame(
    {
        "previous_cell": previous_cell,
        "retained_old": forget_gate * previous_cell,
        "written_new": written_memory,
        "new_cell": new_cell,
    }
)

The additive update is central to improved gradient flow.

# 7. Output Gate and Hidden State

The output gate controls which cell-state information is exposed as the hidden
state.

In [ ]:
output_gate = np.array(
    [0.3, 0.8, 0.6, 0.9]
)

new_hidden = (
    output_gate
    * np.tanh(new_cell)
)

pd.DataFrame(
    {
        "cell_state": new_cell,
        "output_gate": output_gate,
        "hidden_state": new_hidden,
    }
)

The hidden state interacts with the next input and downstream layers. The cell
state is the internal memory path.

# 8. NumPy LSTM Cell

In [ ]:
def lstm_step(
    x_t: np.ndarray,
    h_previous: np.ndarray,
    c_previous: np.ndarray,
    W_x: np.ndarray,
    W_h: np.ndarray,
    bias: np.ndarray,
):
    hidden_dim = h_previous.shape[1]

    gates = (
        x_t @ W_x
        + h_previous @ W_h
        + bias
    )

    forget = sigmoid(
        gates[:, 0:hidden_dim]
    )
    input_gate = sigmoid(
        gates[:, hidden_dim:2 * hidden_dim]
    )
    candidate = np.tanh(
        gates[:, 2 * hidden_dim:3 * hidden_dim]
    )
    output = sigmoid(
        gates[:, 3 * hidden_dim:4 * hidden_dim]
    )

    cell = (
        forget * c_previous
        + input_gate * candidate
    )

    hidden = (
        output
        * np.tanh(cell)
    )

    cache = {
        "forget": forget,
        "input": input_gate,
        "candidate": candidate,
        "output": output,
        "cell": cell,
        "hidden": hidden,
    }

    return hidden, cell, cache

In [ ]:
generator = np.random.default_rng(42)

batch_size = 2
input_dim = 5
hidden_dim = 4

x_t = generator.normal(
    size=(batch_size, input_dim)
)
h_previous = np.zeros(
    (batch_size, hidden_dim)
)
c_previous = np.zeros(
    (batch_size, hidden_dim)
)

W_x_lstm = generator.normal(
    0.0,
    0.2,
    size=(input_dim, 4 * hidden_dim),
)
W_h_lstm = generator.normal(
    0.0,
    0.2,
    size=(hidden_dim, 4 * hidden_dim),
)
b_lstm = np.zeros(
    4 * hidden_dim
)

h_new, c_new, lstm_cache = lstm_step(
    x_t,
    h_previous,
    c_previous,
    W_x_lstm,
    W_h_lstm,
    b_lstm,
)

print("Hidden shape:", h_new.shape)
print("Cell shape:", c_new.shape)

In [ ]:
pd.DataFrame(
    {
        "forget_mean": [lstm_cache["forget"].mean()],
        "input_mean": [lstm_cache["input"].mean()],
        "output_mean": [lstm_cache["output"].mean()],
        "candidate_mean": [lstm_cache["candidate"].mean()],
    }
)

# 9. LSTM Shape Reasoning

For batch size `B`, input dimension `E`, and hidden dimension `H`:

```text
x_t:     (B, E)
h_t:     (B, H)
c_t:     (B, H)
W_x:     (E, 4H)
W_h:     (H, 4H)
bias:    (4H,)
```

Libraries often compute all four gate transformations in one matrix
multiplication for efficiency.

# 10. LSTM Parameter Count

For one LSTM layer:

\[
parameters = 4(EH + H^2 + H)
\]

In [ ]:
def lstm_parameter_count(
    input_dim: int,
    hidden_dim: int,
) -> int:
    return 4 * (
        input_dim * hidden_dim
        + hidden_dim * hidden_dim
        + hidden_dim
    )


parameter_examples = pd.DataFrame(
    [
        (
            100,
            128,
            lstm_parameter_count(100, 128),
        ),
        (
            300,
            256,
            lstm_parameter_count(300, 256),
        ),
        (
            768,
            512,
            lstm_parameter_count(768, 512),
        ),
    ],
    columns=[
        "Input dimension",
        "Hidden dimension",
        "LSTM parameters",
    ],
)

parameter_examples

Embedding and output-layer parameters are additional.

# 11. GRU Overview

A GRU combines memory and hidden state into one vector.

Common equations:

\[
z_t = \sigma(x_tW_{xz} + h_{t-1}W_{hz} + b_z)
\]

\[
r_t = \sigma(x_tW_{xr} + h_{t-1}W_{hr} + b_r)
\]

\[
\tilde{h}_t =
tanh(x_tW_{xh} + (r_t \odot h_{t-1})W_{hh} + b_h)
\]

\[
h_t =
(1-z_t) \odot \tilde{h}_t
+ z_t \odot h_{t-1}
\]

Equation conventions may swap the interpretation of \(z_t\). Always verify the
framework's definition.

# 12. Reset Gate

The reset gate controls how strongly previous hidden information influences
the candidate hidden state.

In [ ]:
previous_hidden = np.array(
    [0.9, -0.6, 0.3, 0.8]
)

reset_gate = np.array(
    [0.1, 0.9, 0.5, 0.2]
)

reset_memory = (
    reset_gate
    * previous_hidden
)

pd.DataFrame(
    {
        "previous_hidden": previous_hidden,
        "reset_gate": reset_gate,
        "gated_previous": reset_memory,
    }
)

# 13. Update Gate

The update gate controls interpolation between old hidden state and candidate
state.

In [ ]:
update_gate = np.array(
    [0.8, 0.1, 0.6, 0.3]
)

candidate_hidden = np.array(
    [-0.2, 0.7, 0.5, -0.4]
)

gru_hidden = (
    update_gate
    * previous_hidden
    + (1.0 - update_gate)
    * candidate_hidden
)

pd.DataFrame(
    {
        "old_hidden": previous_hidden,
        "candidate": candidate_hidden,
        "update_gate": update_gate,
        "new_hidden": gru_hidden,
    }
)

# 14. Candidate Hidden State

The reset gate can suppress irrelevant previous information before the
candidate is calculated.

# 15. NumPy GRU Cell

In [ ]:
def gru_step(
    x_t: np.ndarray,
    h_previous: np.ndarray,
    W_x_z: np.ndarray,
    W_h_z: np.ndarray,
    b_z: np.ndarray,
    W_x_r: np.ndarray,
    W_h_r: np.ndarray,
    b_r: np.ndarray,
    W_x_h: np.ndarray,
    W_h_h: np.ndarray,
    b_h: np.ndarray,
):
    update = sigmoid(
        x_t @ W_x_z
        + h_previous @ W_h_z
        + b_z
    )

    reset = sigmoid(
        x_t @ W_x_r
        + h_previous @ W_h_r
        + b_r
    )

    candidate = np.tanh(
        x_t @ W_x_h
        + (
            reset * h_previous
        ) @ W_h_h
        + b_h
    )

    hidden = (
        update * h_previous
        + (1.0 - update)
        * candidate
    )

    cache = {
        "update": update,
        "reset": reset,
        "candidate": candidate,
        "hidden": hidden,
    }

    return hidden, cache

In [ ]:
W_x_z = generator.normal(
    0.0, 0.2, size=(input_dim, hidden_dim)
)
W_h_z = generator.normal(
    0.0, 0.2, size=(hidden_dim, hidden_dim)
)
b_z = np.zeros(hidden_dim)

W_x_r = generator.normal(
    0.0, 0.2, size=(input_dim, hidden_dim)
)
W_h_r = generator.normal(
    0.0, 0.2, size=(hidden_dim, hidden_dim)
)
b_r = np.zeros(hidden_dim)

W_x_h = generator.normal(
    0.0, 0.2, size=(input_dim, hidden_dim)
)
W_h_h = generator.normal(
    0.0, 0.2, size=(hidden_dim, hidden_dim)
)
b_h = np.zeros(hidden_dim)

gru_new, gru_cache = gru_step(
    x_t,
    h_previous,
    W_x_z,
    W_h_z,
    b_z,
    W_x_r,
    W_h_r,
    b_r,
    W_x_h,
    W_h_h,
    b_h,
)

print("GRU hidden shape:", gru_new.shape)

In [ ]:
pd.DataFrame(
    {
        "update_mean": [gru_cache["update"].mean()],
        "reset_mean": [gru_cache["reset"].mean()],
        "candidate_mean": [gru_cache["candidate"].mean()],
    }
)

# 16. GRU Shape Reasoning

For each of the three transformations:

```text
W_x:  (E, H)
W_h:  (H, H)
bias: (H,)
```

# 17. GRU Parameter Count

For one GRU layer:

\[
parameters = 3(EH + H^2 + H)
\]

In [ ]:
def gru_parameter_count(
    input_dim: int,
    hidden_dim: int,
) -> int:
    return 3 * (
        input_dim * hidden_dim
        + hidden_dim * hidden_dim
        + hidden_dim
    )


pd.DataFrame(
    [
        (
            100,
            128,
            lstm_parameter_count(100, 128),
            gru_parameter_count(100, 128),
        ),
        (
            300,
            256,
            lstm_parameter_count(300, 256),
            gru_parameter_count(300, 256),
        ),
    ],
    columns=[
        "Input dimension",
        "Hidden dimension",
        "LSTM parameters",
        "GRU parameters",
    ],
)

GRUs are usually smaller and faster, but not universally more accurate.

# 18. LSTM Versus GRU

In [ ]:
lstm_gru_comparison = pd.DataFrame(
    [
        (
            "Memory",
            "separate cell and hidden states",
            "single hidden state",
        ),
        (
            "Gates",
            "forget, input, output",
            "reset, update",
        ),
        (
            "Parameters",
            "more",
            "fewer",
        ),
        (
            "Training speed",
            "often slower",
            "often faster",
        ),
        (
            "Best choice",
            "task dependent",
            "task dependent",
        ),
    ],
    columns=["Property", "LSTM", "GRU"],
)

lstm_gru_comparison

Model selection should be empirical and based on validation performance,
stability, latency, and memory.

# 19. Bidirectional Gated RNNs

A bidirectional model processes the sequence in both directions.

Output dimension typically doubles:

```text
unidirectional final state: (B, H)
bidirectional final state:  (B, 2H)
```

Bidirectional models are suitable for complete-input understanding, but not
strict left-to-right generation.

# 20. Multi-Layer Gated RNNs

Stacked recurrent layers pass one layer's sequence outputs to the next layer.

More layers increase capacity and cost. Dropout is commonly applied between
layers.

In [ ]:
stacked_shapes = pd.DataFrame(
    [
        ("Layer 1 input", "(B, T, E)"),
        ("Layer 1 output", "(B, T, H)"),
        ("Layer 2 input", "(B, T, H)"),
        ("Layer 2 output", "(B, T, H)"),
    ],
    columns=["Tensor", "Shape"],
)

stacked_shapes

# 21. Padding, Lengths, and Masks

PyTorch can pack padded sequences so recurrent layers skip padded positions.

In [ ]:
padded_demo = torch.tensor(
    [
        [4, 5, 6, 7],
        [8, 9, 0, 0],
        [2, 3, 1, 0],
    ],
    dtype=torch.long,
)

lengths_demo = torch.tensor(
    [4, 2, 3],
    dtype=torch.long,
)

print("Padded shape:", padded_demo.shape)
print("Lengths:", lengths_demo.tolist())

`enforce_sorted=False` allows unsorted batches when packing.

# 22. Dataset

The dataset contains four balanced classes with varied sequence patterns.

In [ ]:
records = [
    ("doctor treats patient in hospital", "health"),
    ("nurse provides medicine to patient", "health"),
    ("patient visits clinic for diagnosis", "health"),
    ("hospital schedules medical treatment", "health"),
    ("exercise supports long term health", "health"),
    ("nutrition improves patient recovery", "health"),
    ("doctor reviews the medical report", "health"),
    ("clinic provides emergency service", "health"),
    ("nurse helps the patient today", "health"),
    ("medicine reduces the health problem", "health"),
    ("hospital needs experienced doctors", "health"),
    ("patient requests treatment information", "health"),
    ("medical team monitors patient recovery", "health"),
    ("doctor confirms the diagnosis later", "health"),
    ("clinic updates the treatment plan", "health"),
    ("patient receives medicine after examination", "health"),

    ("bank approves customer loan", "finance"),
    ("invoice contains payment charge", "finance"),
    ("customer requests card refund", "finance"),
    ("billing account has a problem", "finance"),
    ("loan interest increased today", "finance"),
    ("bank transfers money safely", "finance"),
    ("payment failed on the card", "finance"),
    ("refund request remains pending", "finance"),
    ("invoice price is incorrect", "finance"),
    ("customer updates bank account", "finance"),
    ("billing service changed the charge", "finance"),
    ("loan payment needs approval", "finance"),
    ("bank reviews the financial request", "finance"),
    ("customer receives refund after review", "finance"),
    ("payment system confirms the transaction", "finance"),
    ("card account shows an extra charge", "finance"),

    ("software update caused an error", "technology"),
    ("application cannot reach the server", "technology"),
    ("network upload failed today", "technology"),
    ("computer needs a system update", "technology"),
    ("device cannot install the software", "technology"),
    ("server lost important data", "technology"),
    ("application displays a network error", "technology"),
    ("computer connects to the server", "technology"),
    ("upload request failed again", "technology"),
    ("system update needs technical help", "technology"),
    ("device reports a software problem", "technology"),
    ("network service is unavailable", "technology"),
    ("server restarts after the update", "technology"),
    ("application recovers after installation", "technology"),
    ("computer stores data on server", "technology"),
    ("network error interrupts the upload", "technology"),

    ("flight arrives at airport", "travel"),
    ("tourist books hotel reservation", "travel"),
    ("airport lost passenger luggage", "travel"),
    ("travel ticket changed today", "travel"),
    ("flight delay affects the journey", "travel"),
    ("hotel reservation needs an update", "travel"),
    ("tourist visits the city museum", "travel"),
    ("beach trip starts tomorrow", "travel"),
    ("airport changes the flight gate", "travel"),
    ("passenger requests travel information", "travel"),
    ("journey includes a hotel stay", "travel"),
    ("ticket service reports a delay", "travel"),
    ("tourist reaches airport before departure", "travel"),
    ("passenger collects luggage after arrival", "travel"),
    ("hotel confirms the reservation", "travel"),
    ("flight continues after a short delay", "travel"),
]

dataset = pd.DataFrame(
    records,
    columns=["text", "label"],
)

dataset["label"].value_counts()

# 23. Train, Validation, and Test Splits

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    dataset["text"],
    dataset["label"],
    test_size=0.25,
    random_state=42,
    stratify=dataset["label"],
)

X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full,
)

print("Training:", len(X_train))
print("Validation:", len(X_validation))
print("Test:", len(X_test))

# 24. Vocabulary and Encoding

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(
        text.lower()
    )


training_counts = Counter(
    token
    for text in X_train
    for token in tokenize(text)
)

vocabulary = [
    "<PAD>",
    "<UNK>",
] + sorted(training_counts)

word_to_index = {
    word: index
    for index, word in enumerate(vocabulary)
}

PAD_ID = word_to_index["<PAD>"]
UNK_ID = word_to_index["<UNK>"]

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

print("Vocabulary size:", len(vocabulary))
print("Classes:", label_encoder.classes_.tolist())

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = label_encoder.transform(
            list(labels)
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        token_ids = [
            word_to_index.get(
                token,
                UNK_ID,
            )
            for token in tokenize(
                self.texts[index]
            )
        ]

        return (
            torch.tensor(
                token_ids,
                dtype=torch.long,
            ),
            torch.tensor(
                self.labels[index],
                dtype=torch.long,
            ),
            self.texts[index],
        )

# 25. DataLoaders and Collation

In [ ]:
def collate_batch(batch):
    sequences, labels, texts = zip(*batch)

    lengths = torch.tensor(
        [len(sequence) for sequence in sequences],
        dtype=torch.long,
    )

    max_length = int(
        lengths.max().item()
    )

    padded = torch.full(
        (
            len(sequences),
            max_length,
        ),
        fill_value=PAD_ID,
        dtype=torch.long,
    )

    for row, sequence in enumerate(sequences):
        padded[row, :len(sequence)] = sequence

    return (
        padded,
        lengths,
        torch.stack(labels),
        list(texts),
    )


train_dataset = TextDataset(
    X_train,
    y_train,
)
validation_dataset = TextDataset(
    X_validation,
    y_validation,
)
test_dataset = TextDataset(
    X_test,
    y_test,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(42),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_batch,
)

batch_tokens, batch_lengths, batch_labels, batch_texts = next(
    iter(train_loader)
)

print("Tokens:", batch_tokens.shape)
print("Lengths:", batch_lengths.shape)
print("Labels:", batch_labels.shape)

# 26. LSTM Text Classifier

In [ ]:
class GatedRNNClassifier(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        embedding_dim: int,
        hidden_dim: int,
        class_count: int,
        cell_type: str = "lstm",
        bidirectional: bool = False,
        num_layers: int = 1,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.cell_type = cell_type
        self.bidirectional = bidirectional
        self.num_directions = (
            2 if bidirectional else 1
        )

        self.embedding = nn.Embedding(
            vocabulary_size,
            embedding_dim,
            padding_idx=PAD_ID,
        )

        recurrent_dropout = (
            dropout if num_layers > 1 else 0.0
        )

        if cell_type == "lstm":
            self.recurrent = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=recurrent_dropout,
            )
        elif cell_type == "gru":
            self.recurrent = nn.GRU(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=recurrent_dropout,
            )
        else:
            raise ValueError(
                "cell_type must be 'lstm' or 'gru'"
            )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_dim
            * self.num_directions,
            class_count,
        )

    def forward(
        self,
        token_ids: torch.Tensor,
        lengths: torch.Tensor,
    ):
        embedded = self.embedding(token_ids)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        if self.cell_type == "lstm":
            _, (hidden, cell) = self.recurrent(
                packed
            )
        else:
            _, hidden = self.recurrent(
                packed
            )
            cell = None

        if self.bidirectional:
            representation = torch.cat(
                (
                    hidden[-2],
                    hidden[-1],
                ),
                dim=1,
            )
        else:
            representation = hidden[-1]

        logits = self.classifier(
            self.dropout(
                representation
            )
        )

        return {
            "logits": logits,
            "representation": representation,
            "hidden": hidden,
            "cell": cell,
        }

In [ ]:
torch.manual_seed(42)

lstm_model = GatedRNNClassifier(
    vocabulary_size=len(vocabulary),
    embedding_dim=24,
    hidden_dim=20,
    class_count=len(label_encoder.classes_),
    cell_type="lstm",
    dropout=0.2,
)

sample_output = lstm_model(
    batch_tokens,
    batch_lengths,
)

print("Logits:", sample_output["logits"].shape)
print(
    "Representation:",
    sample_output["representation"].shape,
)

# 27. GRU Text Classifier

In [ ]:
torch.manual_seed(42)

gru_model = GatedRNNClassifier(
    vocabulary_size=len(vocabulary),
    embedding_dim=24,
    hidden_dim=20,
    class_count=len(label_encoder.classes_),
    cell_type="gru",
    dropout=0.2,
)

gru_output = gru_model(
    batch_tokens,
    batch_lengths,
)

print("GRU logits:", gru_output["logits"].shape)

The classifiers differ only in recurrent cell type, enabling a controlled
comparison.

# 28. Training Utilities

In [ ]:
DEVICE = torch.device("cpu")


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    loss_function: nn.Module,
):
    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []
    all_texts = []
    total_loss = 0.0
    total_examples = 0

    with torch.no_grad():
        for token_ids, lengths, labels, texts in loader:
            token_ids = token_ids.to(DEVICE)
            lengths = lengths.to(DEVICE)
            labels = labels.to(DEVICE)

            output = model(
                token_ids,
                lengths,
            )

            logits = output["logits"]
            loss = loss_function(
                logits,
                labels,
            )

            probabilities = torch.softmax(
                logits,
                dim=1,
            )
            predictions = probabilities.argmax(
                dim=1
            )

            total_loss += (
                loss.item()
                * len(labels)
            )
            total_examples += len(labels)

            all_labels.extend(
                labels.cpu().tolist()
            )
            all_predictions.extend(
                predictions.cpu().tolist()
            )
            all_probabilities.extend(
                probabilities.cpu().tolist()
            )
            all_texts.extend(texts)

    return {
        "loss": total_loss / total_examples,
        "accuracy": accuracy_score(
            all_labels,
            all_predictions,
        ),
        "macro_f1": f1_score(
            all_labels,
            all_predictions,
            average="macro",
        ),
        "labels": np.asarray(all_labels),
        "predictions": np.asarray(all_predictions),
        "probabilities": np.asarray(all_probabilities),
        "texts": all_texts,
    }

In [ ]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    validation_loader: DataLoader,
    epochs: int = 45,
    learning_rate: float = 0.01,
    weight_decay: float = 1e-4,
    clip_norm: float = 5.0,
    patience: int = 8,
):
    model = model.to(DEVICE)

    loss_function = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )
    best_validation_loss = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        total_examples = 0
        gradient_norms = []

        for token_ids, lengths, labels, _ in train_loader:
            token_ids = token_ids.to(DEVICE)
            lengths = lengths.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            output = model(
                token_ids,
                lengths,
            )

            loss = loss_function(
                output["logits"],
                labels,
            )

            loss.backward()

            gradient_norm = clip_grad_norm_(
                model.parameters(),
                max_norm=clip_norm,
            )

            gradient_norms.append(
                float(gradient_norm)
            )

            optimizer.step()

            total_loss += (
                loss.item()
                * len(labels)
            )
            total_examples += len(labels)

        validation_metrics = evaluate_model(
            model,
            validation_loader,
            loss_function,
        )

        history.append(
            {
                "epoch": epoch,
                "train_loss": (
                    total_loss
                    / total_examples
                ),
                "validation_loss": validation_metrics[
                    "loss"
                ],
                "validation_accuracy": validation_metrics[
                    "accuracy"
                ],
                "validation_macro_f1": validation_metrics[
                    "macro_f1"
                ],
                "mean_gradient_norm": float(
                    np.mean(
                        gradient_norms
                    )
                ),
            }
        )

        if (
            validation_metrics["loss"]
            < best_validation_loss - 1e-5
        ):
            best_validation_loss = validation_metrics[
                "loss"
            ]

            best_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= patience
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
    )

# 29. Training the LSTM

In [ ]:
set_seed(42)

lstm_model = GatedRNNClassifier(
    vocabulary_size=len(vocabulary),
    embedding_dim=24,
    hidden_dim=20,
    class_count=len(label_encoder.classes_),
    cell_type="lstm",
    dropout=0.2,
)

trained_lstm, lstm_history = train_model(
    lstm_model,
    train_loader,
    validation_loader,
)

print("LSTM epochs:", len(lstm_history))
print(
    "Best validation F1:",
    round(
        lstm_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    lstm_history["epoch"],
    lstm_history["train_loss"],
    label="Training loss",
)
plt.plot(
    lstm_history["epoch"],
    lstm_history["validation_loss"],
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

# 30. Training the GRU

In [ ]:
set_seed(42)

gru_model = GatedRNNClassifier(
    vocabulary_size=len(vocabulary),
    embedding_dim=24,
    hidden_dim=20,
    class_count=len(label_encoder.classes_),
    cell_type="gru",
    dropout=0.2,
)

trained_gru, gru_history = train_model(
    gru_model,
    train_loader,
    validation_loader,
)

print("GRU epochs:", len(gru_history))
print(
    "Best validation F1:",
    round(
        gru_history[
            "validation_macro_f1"
        ].max(),
        3,
    ),
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    gru_history["epoch"],
    gru_history["train_loss"],
    label="Training loss",
)
plt.plot(
    gru_history["epoch"],
    gru_history["validation_loss"],
    label="Validation loss",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GRU Learning Curves")
plt.legend()
plt.tight_layout()
plt.show()

# 31. Comparing Models

In [ ]:
loss_function = nn.CrossEntropyLoss()

lstm_validation = evaluate_model(
    trained_lstm,
    validation_loader,
    loss_function,
)

gru_validation = evaluate_model(
    trained_gru,
    validation_loader,
    loss_function,
)

comparison = pd.DataFrame(
    [
        (
            "LSTM",
            sum(
                parameter.numel()
                for parameter in trained_lstm.parameters()
            ),
            lstm_validation["accuracy"],
            lstm_validation["macro_f1"],
        ),
        (
            "GRU",
            sum(
                parameter.numel()
                for parameter in trained_gru.parameters()
            ),
            gru_validation["accuracy"],
            gru_validation["macro_f1"],
        ),
    ],
    columns=[
        "Model",
        "Trainable parameters",
        "Validation accuracy",
        "Validation macro F1",
    ],
)

comparison

The dataset is intentionally small. Differences should not be generalized
beyond this demonstration.

# 32. Evaluation and Confusion Matrices

In [ ]:
lstm_test = evaluate_model(
    trained_lstm,
    test_loader,
    loss_function,
)

gru_test = evaluate_model(
    trained_gru,
    test_loader,
    loss_function,
)

test_comparison = pd.DataFrame(
    [
        (
            "LSTM",
            lstm_test["loss"],
            lstm_test["accuracy"],
            lstm_test["macro_f1"],
        ),
        (
            "GRU",
            gru_test["loss"],
            gru_test["accuracy"],
            gru_test["macro_f1"],
        ),
    ],
    columns=[
        "Model",
        "Test loss",
        "Test accuracy",
        "Test macro F1",
    ],
)

test_comparison.round(3)

In [ ]:
best_name = (
    "LSTM"
    if lstm_validation["macro_f1"]
    >= gru_validation["macro_f1"]
    else "GRU"
)

best_model = (
    trained_lstm
    if best_name == "LSTM"
    else trained_gru
)

best_test = (
    lstm_test
    if best_name == "LSTM"
    else gru_test
)

predicted_labels = label_encoder.inverse_transform(
    best_test["predictions"]
)
actual_labels = label_encoder.inverse_transform(
    best_test["labels"]
)

print("Selected model:", best_name)
print(
    classification_report(
        actual_labels,
        predicted_labels,
        zero_division=0,
    )
)

In [ ]:
class_names = list(
    label_encoder.classes_
)

matrix = confusion_matrix(
    actual_labels,
    predicted_labels,
    labels=class_names,
)

pd.DataFrame(
    matrix,
    index=[
        f"actual_{label}"
        for label in class_names
    ],
    columns=[
        f"predicted_{label}"
        for label in class_names
    ],
)

# 33. Error Analysis

In [ ]:
error_frame = pd.DataFrame(
    {
        "text": best_test["texts"],
        "actual": actual_labels,
        "predicted": predicted_labels,
        "confidence": best_test[
            "probabilities"
        ].max(axis=1),
    }
)

error_frame["correct"] = (
    error_frame["actual"]
    == error_frame["predicted"]
)

error_frame.sort_values(
    ["correct", "confidence"],
    ascending=[True, True],
)

Review errors for:

- unknown words;
- mixed domains;
- long-distance dependencies;
- short ambiguous inputs;
- truncation;
- label overlap;
- overconfident wrong predictions.

# 34. Bidirectional Model

In [ ]:
set_seed(42)

bidirectional_gru = GatedRNNClassifier(
    vocabulary_size=len(vocabulary),
    embedding_dim=24,
    hidden_dim=16,
    class_count=len(label_encoder.classes_),
    cell_type="gru",
    bidirectional=True,
    dropout=0.2,
)

bidirectional_output = bidirectional_gru(
    batch_tokens,
    batch_lengths,
)

print(
    "Bidirectional representation:",
    bidirectional_output[
        "representation"
    ].shape,
)

Bidirectionality doubles the representation size when forward and backward
states are concatenated.

# 35. Representation Inspection

In [ ]:
best_model.eval()

with torch.no_grad():
    token_ids, lengths, labels, texts = next(
        iter(test_loader)
    )

    output = best_model(
        token_ids,
        lengths,
    )

    representations = output[
        "representation"
    ].cpu().numpy()

representation_frame = pd.DataFrame(
    {
        "text": texts,
        "label": label_encoder.inverse_transform(
            labels.numpy()
        ),
        "representation_norm": np.linalg.norm(
            representations,
            axis=1,
        ),
    }
)

representation_frame

Representation norms and distances can be inspected, but individual hidden
dimensions are not automatically interpretable.

# 36. Gate Interpretation

Gate values are internal control signals, not direct linguistic explanations.

A high forget gate may indicate retention, but interpretation requires:

- comparison across examples;
- time-step alignment;
- counterfactual tests;
- stability across seeds.

In [ ]:
gate_interpretation = pd.DataFrame(
    [
        ("Forget gate", "retention of previous cell memory"),
        ("Input gate", "writing candidate memory"),
        ("Output gate", "exposing cell information"),
        ("Reset gate", "using previous hidden state in candidate"),
        ("Update gate", "interpolating old and candidate state"),
    ],
    columns=["Gate", "Operational role"],
)

gate_interpretation

# 37. Gradient Clipping and Stability

In [ ]:
gradient_history = pd.DataFrame(
    {
        "LSTM": lstm_history[
            "mean_gradient_norm"
        ],
        "GRU": gru_history[
            "mean_gradient_norm"
        ].reindex(
            range(
                max(
                    len(lstm_history),
                    len(gru_history),
                )
            )
        ),
    }
)

gradient_history.describe()

Pre-clipping norms reveal whether clipping is frequently active. Excessive
clipping may indicate an overly large learning rate or unstable model.

# 38. Dropout and Regularization

Useful regularization methods include:

- embedding dropout;
- recurrent-layer dropout in stacked models;
- output dropout;
- weight decay;
- early stopping;
- smaller hidden dimensions.

In [ ]:
regularization_summary = pd.DataFrame(
    [
        ("Embedding dropout", "token representations"),
        ("Inter-layer dropout", "stacked recurrent outputs"),
        ("Output dropout", "final representation"),
        ("Weight decay", "large parameters"),
        ("Early stopping", "validation overfitting"),
    ],
    columns=["Method", "Primary target"],
)

regularization_summary

PyTorch recurrent dropout is applied between stacked layers, not across time
within a single-layer cell.

# 39. Computational Cost

Gated models remain sequential across time.

Approximate per-step cost depends on:

- embedding dimension;
- hidden dimension;
- number of gates;
- layers;
- directions;
- batch size.

In [ ]:
cost_comparison = pd.DataFrame(
    [
        ("Vanilla RNN", 1, "lowest"),
        ("GRU", 3, "medium"),
        ("LSTM", 4, "highest"),
        ("Bidirectional LSTM", 8, "very high"),
    ],
    columns=[
        "Architecture",
        "Relative gate transformations",
        "Relative recurrent cost",
    ],
)

cost_comparison

# 40. Common Failure Modes

- insufficient data;
- unstable learning rate;
- overfitting;
- ignored sequence lengths;
- incorrect final-state extraction;
- excessive padding;
- exploding gradients;
- weak OOV handling;
- misleading gate interpretation;
- assuming LSTM or GRU is always superior.

In [ ]:
failure_modes = pd.DataFrame(
    [
        ("Padding contamination", "pack sequences or apply masks"),
        ("Exploding gradients", "clip gradient norms"),
        ("Overfitting", "dropout, weight decay, early stopping"),
        ("Long sequences", "chunking, attention, or Transformers"),
        ("OOV words", "subword tokenization"),
        ("Slow training", "smaller models or parallel architectures"),
    ],
    columns=["Failure", "Response"],
)

failure_modes

# 41. Arabic and Multilingual Considerations

Arabic gated sequence models are affected by:

- clitic attachment;
- rich morphology;
- optional diacritics;
- orthographic variation;
- MSA and dialects;
- code-switching;
- right-to-left display;
- tokenization granularity.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            "وَسَيَكْتُبُونَهَا",
            "وَ + سَ + يَكْتُبُونَ + هَا",
        ),
        (
            "وَبِالْمَدْرَسَةِ",
            "وَ + بِ + الْمَدْرَسَةِ",
        ),
        (
            "كِتَابُهُمَا",
            "كِتَابُ + هُمَا",
        ),
    ],
    columns=[
        "Fully vocalized surface form",
        "Illustrative segmentation",
    ],
)

arabic_examples

Tokenization choice changes both sequence length and vocabulary sparsity.

For fully vocalized Arabic tasks, tashkeel may encode lexical and
morphological information and should not be removed automatically.

In [ ]:
arabic_sequence_units = pd.DataFrame(
    [
        ("Word", "shorter sequences", "high sparsity"),
        ("Morphological segment", "linguistic units", "analyzer required"),
        ("Subword", "balanced coverage", "fragmentation possible"),
        ("Character", "small vocabulary", "very long sequences"),
    ],
    columns=["Unit", "Benefit", "Cost"],
)

arabic_sequence_units

Multilingual gated models also face imbalanced corpora, different scripts, and
unequal sequence lengths across languages.

# 42. Reproducibility and Reporting

Report:

- dataset and split;
- tokenizer and normalization;
- vocabulary size;
- sequence-length distribution;
- embedding dimension;
- hidden dimension;
- LSTM or GRU;
- layers and directions;
- dropout;
- optimizer and learning rate;
- weight decay;
- gradient clipping;
- early stopping;
- random seed;
- evaluation metrics;
- hardware.

In [ ]:
import platform

metadata = pd.Series(
    {
        "dataset_examples": len(dataset),
        "classes": dataset["label"].nunique(),
        "vocabulary_size": len(vocabulary),
        "embedding_dimension": 24,
        "hidden_dimension": 20,
        "batch_size": 8,
        "optimizer": "Adam",
        "gradient_clip_norm": 5.0,
        "device": str(DEVICE),
        "random_seed": 42,
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
        "torch_version": torch.__version__,
    },
    name="Gated RNN experiment",
)

metadata

# 43. Knowledge Check

1. Why do vanilla RNNs struggle with long-range dependencies?
2. What is the LSTM cell state?
3. What does the forget gate control?
4. How do the input gate and candidate memory interact?
5. What does the output gate control?
6. Why is the cell-state update additive?
7. What are the GRU reset and update gates?
8. How do LSTM and GRU parameter counts differ?
9. Why pack padded sequences?
10. How is a bidirectional representation constructed?
11. Why is bidirectionality unsuitable for strict causal generation?
12. What does gradient clipping prevent?
13. Where does PyTorch recurrent dropout apply?
14. Why should gate values not be treated as direct explanations?
15. Which Arabic properties affect gated sequence models?

# 44. Exercises

## Exercise 1 — LSTM Gates

Calculate one LSTM update manually.

## Exercise 2 — Forget Bias

Compare initial forget biases of 0 and 1.

## Exercise 3 — GRU Gates

Calculate one GRU update manually.

## Exercise 4 — Parameter Counts

Compare RNN, GRU, and LSTM parameter counts.

## Exercise 5 — Model Comparison

Compare LSTM and GRU across several hidden dimensions.

## Exercise 6 — Bidirectionality

Train a bidirectional GRU and compare performance and parameter count.

## Exercise 7 — Stacking

Compare one-layer and two-layer gated models.

## Exercise 8 — Pooling

Compare final hidden state with mean pooling over sequence outputs.

## Exercise 9 — Arabic Classification

Compare word, segmented, subword, and character inputs.

## Exercise 10 — Long Dependencies

Create synthetic examples where a cue appears far from the target.

## Challenge Exercises

1. Implement LSTM backpropagation with NumPy.
2. Implement GRU backpropagation with NumPy.
3. Extract and visualize gate values across time.
4. Add class-weighted loss and calibration.
5. Build a sequence-labeling BiLSTM model.

# 45. Summary and Next Lesson

In this lesson:

- gated recurrent networks were motivated by long-range dependency problems;
- LSTM hidden and cell states were distinguished;
- forget, input, candidate, and output transformations were explained;
- one LSTM step was implemented with NumPy;
- GRU reset and update mechanisms were explained;
- one GRU step was implemented with NumPy;
- parameter counts and computational costs were compared;
- packed sequences prevented padded positions from affecting recurrence;
- executable LSTM and GRU classifiers were trained with PyTorch;
- gradient clipping, dropout, weight decay, and early stopping improved
  stability;
- bidirectional and stacked models were introduced;
- evaluation, error analysis, and representation inspection were performed;
- Arabic morphology, clitics, and tashkeel were connected to sequence design.

## Next Lesson

**Lesson 29: Sequence-to-Sequence Learning with Encoder–Decoder Networks**
introduces neural encoding, autoregressive decoding, teacher forcing,
sequence loss, inference, and the limitations that motivate attention.

# References

- Hochreiter, S., & Schmidhuber, J. *Long Short-Term Memory*.
- Cho, K. et al. GRU and encoder–decoder literature.
- Goodfellow, I., Bengio, Y., & Courville, A. *Deep Learning*.
- Goldberg, Y. *Neural Network Methods for Natural Language Processing*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.